In [4]:
from typing import Any

from langchain_community.embeddings import DashScopeEmbeddings
from pymilvus import MilvusClient

from app.core.config import settings

embedding = DashScopeEmbeddings(model='text-embedding-v4')
client = MilvusClient(uri=settings.rag.milvus_url)

#稠密向量 语义搜索方法
def dense_search(query:str):
    query_vector = embedding.embed_query(query)
    res = client.search(
        collection_name="my_1",
        anns_field='dense',
        data=[query_vector],
        search_params={"metric_type":"COSINE"},
        output_fields=["h1",'h2','h3','text'],
        limit=3
    )
    for hits in res:
        for hit in hits:
            print("==============================")
            print(hit)
#稀疏搜索方法使用BM25算法
def sparse_search(query:str)->Any:
    res = client.search(
        collection_name = "my_1",
        data = [query],
        anns_field= "sparse",
        search_params={"metric_type":"BM25"},
        limit = 3,
        output_fields=["h1","h2","h3","text"]
    )
    for hits in res:
        for hit in hits:
            print(hit)

#稠密向量
print("===============稠密向量检索：语义相似性查询===============")
dense_search("教育的负面作用是什么")
print("\n===============稠密向量检索：专有名词查询===============")
dense_search("给教师的建议是谁写的")
#稀疏向量
print("---------------稀疏向量检索：语义相似性查询---------------")
sparse_search("教育的负面作用是什么")
print("\n---------------稀疏向量检索：专有名词查询---------------")
sparse_search("给教师的建议是谁写的")

===============稠密向量检索：语义相似性查询===============
{'id': 15, 'distance': 0.6111685633659363, 'entity': {'h3': '（一）个体发展功能和社会发展功能', 'text': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。', 'h1': '第二章 教育基本原理', 'h2': '第一节 教育的功能'}}

===============稠密向量检索：专有名词查询===============
{'id': 1, 'distance': 0.47179198265075684, 'entity': {'h1': '第一章 教育概述', 'h2': '第一节 中外教育家及其教育思想', 'h3': '（一）教育学萌芽时期代表作', 'text': '# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。'}}
{'id': 8, 'distance': 0.46489983797073364, 'entity': {'h1': '第一章 教育概述', 'h2': '第一节 中外教育家及其教育思想', 'h3': '（八）教育学分化时期代表人物及主要思想', 'text': '### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**